# SD-MoSE: Data Pipeline

This notebook orchestrates the data downloading and preprocessing pipeline for the Soft Regime Mixture of Symbolic Experts (SD-MoSE) project.

## Steps
1. **Setup**: Install dependencies.
2. **Download**: Fetch SOCAT (Physics) and Copernicus (Biology) data.
3. **Preprocess**: Align, merge, split (Train/Test), and normalize data.
4. **Verify**: Inspect the generated datasets.

In [ ]:
# 1. Setup
import sys
import os
from pathlib import Path

# Add project root to path to import scripts
sys.path.append(os.path.abspath('..'))

import xarray as xr
import matplotlib.pyplot as plt
from scripts.download import check_copernicus_auth, download_socat, download_chlorophyll
from scripts.preprocess import preprocess, TRAIN_OUTPUT_PATH, TEST_OUTPUT_PATH

## 2. Download Data
Ensures raw data is present in `data/01_raw`. Requires `copernicusmarine` authentication.

In [ ]:
DATA_DIR = Path("../data/01_raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Check Auth
check_copernicus_auth()

# Download
download_socat(DATA_DIR)
download_chlorophyll(DATA_DIR)

## 3. Preprocess Data
Runs the `preprocess.py` logic:
- Time slice: 2015-2024
- Train: 2015-2021
- Test: 2022-2024
- Spatial Regridding & Normalization

In [ ]:
preprocess()

## 4. Verification
Check shapes and inspect a sample frame.

In [ ]:
# Load processed data
try:
    ds_train = xr.open_dataset(TRAIN_OUTPUT_PATH)
    ds_test = xr.open_dataset(TEST_OUTPUT_PATH)

    print("Train Shape:", ds_train.dims)
    print("Test Shape:", ds_test.dims)
    print("\nVariables:", list(ds_train.data_vars))
    
    # Quick Plot of first time step SST
    plt.figure(figsize=(10, 5))
    ds_train['sst'].isel(time=0).plot(cmap='coolwarm')
    plt.title("Sample SST (Normalized) - Train Set")
    plt.show()
    
except FileNotFoundError:
    print("❌ Processed files not found. Run the steps above.")